In [1]:
import json
import logging
import random
import time
import uuid
from datetime import datetime, UTC
from typing import Optional, Dict

from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka.error import KafkaException
from faker import Faker

In [2]:
def get_admin(kafka_config: dict) -> AdminClient:
    return AdminClient(kafka_config)


def list_topics(admin: AdminClient, timeout: float = 10.0):
    md = admin.list_topics(timeout=timeout)
    topics = md.topics.keys()
    for t in sorted(topics):
        print(t)


def create_topic(admin: AdminClient, topic_name: str, partitions: int = 1, replication_factor: int = 1, timeout: float = 30.0):
    new_topic = NewTopic(topic_name, num_partitions=partitions, replication_factor=replication_factor)
    fs = admin.create_topics([new_topic], request_timeout=timeout)
    try:
        fs[topic_name].result()
        logging.info(f"created topic: {topic_name}")
    except KafkaException as e:
        msg = str(e)
        if "TOPIC_ALREADY_EXISTS" in msg or "TopicExistsError" in msg:
            logging.info(f"topic already exists: {topic_name}")
        else:
            raise RuntimeError(f"create_topics failed for {topic_name}: {e}") from e


def delete_topics(admin: AdminClient, topics, timeout: float = 30.0) -> None:
    if isinstance(topics, str):
        topics = [topics]
    topics = list(topics)
    logging.info("Deleting topics:", topics)
    fs = admin.delete_topics(topics, request_timeout=timeout)
    for t in topics:
        try:
            fs[t].result()
            logging.info(f"deleted topic: {t}")
        except KafkaException as e:
            raise RuntimeError(f"delete_topics failed for {t}: {e}") from e


def delete_consumer_group(admin: AdminClient, group_id: str, timeout: float = 10.0) -> None:
    futures = admin.delete_consumer_groups([group_id], request_timeout=timeout)

    f = futures[group_id]
    try:
        f.result(timeout=timeout)
        logging.info(f"Deleted consumer group: {group_id}")
    except Exception as e:
        logging.info(f"Failed to delete consumer group: {group_id}\n   {type(e).__name__}: {e}")


def _delivery_report(err, msg) -> None:
    if err is not None:
        logging.info(f"Delivery failed: {err}")
    else:
        logging.info(f"Delivered to {msg.topic()} [{msg.partition()}] @ offset {msg.offset()}")

In [3]:
_kafka_config = {
    "bootstrap.servers": "confluent-kafka-broker:9093",
    "security.protocol": "SASL_PLAINTEXT",
    "sasl.mechanism": "PLAIN",
    "sasl.username": "admin",
    "sasl.password": "admin"
}

_topic_name = "mmix-products-topic"

In [17]:
list_topics(get_admin(_kafka_config))
#create_topic(admin=get_admin(_kafka_config), topic_name=_topic_name)
#delete_consumer_group(get_admin(_kafka_config), "debezium-mysql-source-flink")

__consumer_offsets
__internal_confluent_only_broker_info
_confluent-command
_confluent-controlcenter-7-7-7-1-AlertHistoryStore-changelog
_confluent-controlcenter-7-7-7-1-AlertHistoryStore-repartition
_confluent-controlcenter-7-7-7-1-Group-ONE_MINUTE-changelog
_confluent-controlcenter-7-7-7-1-Group-ONE_MINUTE-repartition
_confluent-controlcenter-7-7-7-1-Group-THREE_HOURS-changelog
_confluent-controlcenter-7-7-7-1-Group-THREE_HOURS-repartition
_confluent-controlcenter-7-7-7-1-KSTREAM-OUTEROTHER-0000000106-store-changelog
_confluent-controlcenter-7-7-7-1-KSTREAM-OUTEROTHER-0000000106-store-repartition
_confluent-controlcenter-7-7-7-1-KSTREAM-OUTERTHIS-0000000105-store-changelog
_confluent-controlcenter-7-7-7-1-KSTREAM-OUTERTHIS-0000000105-store-repartition
_confluent-controlcenter-7-7-7-1-MetricsAggregateStore-changelog
_confluent-controlcenter-7-7-7-1-MetricsAggregateStore-repartition
_confluent-controlcenter-7-7-7-1-MonitoringMessageAggregatorWindows-ONE_MINUTE-changelog
_confluent-cont

In [16]:
delete_topics(get_admin(_kafka_config), "")

In [21]:
delete_consumer_group(get_admin(_kafka_config), "connect-s3-sync-connector")